# Task 102: pypetal_reverb — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: AGN reverberation mapping using cross-correlation and deconvolution

**Paper**: pyPETaL: A Pipeline for Estimating AGN Time Lags
**Repository**: https://github.com/Zstone19/pypetal

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 25.36 dB ← 🔧 修复前: 2.61 dB
- **SSIM**: N/A (1D)
- **Evaluation**: AGN混响映射 — 传递函数反卷积

### Mathematical Formulation

The imaging process is modeled as a convolution with the Point Spread Function (PSF):

$$y = h \ast x + \eta$$

where $x$ is the true image, $h$ is the PSF, $\ast$ denotes convolution, and $\eta \sim \mathcal{N}(0, \sigma^2)$ is additive noise.

**Inverse problem** — recover $x$ by solving:

$$\hat{x} = \arg\min_x \frac{1}{2}\|h \ast x - y\|_2^2 + \lambda \mathcal{R}(x)$$

Common regularizers $\mathcal{R}(x)$:
- **Tikhonov**: $\mathcal{R}(x) = \|x\|_2^2$
- **Total Variation**: $\mathcal{R}(x) = \|\nabla x\|_1$
- **Sparsity (L1)**: $\mathcal{R}(x) = \|x\|_1$

**Richardson-Lucy** iteration (for Poisson noise):
$$x^{(k+1)} = x^{(k)} \cdot \left( h^T \ast \frac{y}{h \ast x^{(k)}} \right)$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
pypetal_reverb - Reverberation Mapping (AGN Transfer Function Recovery)
========================================================================
From AGN continuum and emission line light curves, recover the
transfer function Ψ(τ) via Tikhonov deconvolution.

Physics:
  - Forward:  L(t) = ∫ Ψ(τ) × C(t-τ) dτ + noise   (convolution)
  - GT transfer function: Gaussian peaked at τ = 20 days
  - Inverse: Tikhonov-regularised deconvolution in frequency domain
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_ccf`, `main`

In [ ]:
def compute_ccf(continuum, line_obs, dt):
    """
    Cross-correlation function between continuum and line.
    """
    N = len(continuum)
    c_norm = continuum - continuum.mean()
    l_norm = line_obs - line_obs.mean()
    ccf = np.correlate(l_norm, c_norm, mode='full')
    ccf = ccf / (np.std(continuum) * np.std(line_obs) * N)
    lags = np.arange(-N + 1, N) * dt
    # Keep only positive lags
    pos = lags >= 0
    return lags[pos], ccf[pos]

# ====================================================================
# 6. Main
# ====================================================================
def main():
    print("=" * 60)
    print("Task 102: Reverberation Mapping (pypetal_reverb)")
    print("=" * 60)

    t0 = time.time()

    # Generate data
    print("\n[1] Generating AGN light curves ...")
    t, continuum = create_continuum(N_TIME, DT)
    tau, psi_gt = create_transfer_function(N_TIME, DT, TAU_PEAK, TAU_SIGMA)
    line_clean, line_obs, noise_std = forward_model(continuum, psi_gt, DT, SNR)
    print(f"    Time span: {T_MAX:.0f} days, {N_TIME} samples")
    print(f"    Noise std: {noise_std:.4f}")

    # Inverse
    print("[2] Tikhonov deconvolution ...")
    psi_rec = tikhonov_deconvolve(continuum, line_obs, DT, lam=LAMBDA_REG)

    # CCF
    print("[3] Computing CCF ...")
    ccf_lags, ccf = compute_ccf(continuum, line_obs, DT)

    elapsed = time.time() - t0
    print(f"    Elapsed: {elapsed:.1f} s")

    # Metrics
    print("[4] Computing metrics ...")
    psnr, cc, rmse, peak_err, psi_rec_norm, psi_gt_trimmed = \
        compute_metrics(psi_gt, psi_rec, tau, DT)

    print(f"    PSNR = {psnr:.2f} dB")
    print(f"    CC   = {cc:.4f}")
    print(f"    RMSE = {rmse:.6f}")
    print(f"    Peak lag error = {peak_err:.1f} days")

    metrics = {
        "PSNR": float(psnr),
        "CC": float(cc),
        "RMSE": float(rmse),
        "peak_lag_error": float(peak_err),
    }

    # Save
    print("[5] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), psi_gt)
        np.save(os.path.join(d, "recon_output.npy"), psi_rec)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # Plot
    print("[6] Plotting ...")
    plot_results(t, continuum, line_clean, line_obs, tau, psi_gt,
                 psi_rec_norm, ccf_lags, ccf, metrics)

    print(f"\n{'='*60}")
    print("Task 102 COMPLETE")
    print(f"{'='*60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `create_continuum`, `create_transfer_function`, `forward_model`, `tikhonov_deconvolve`

In [ ]:
# ====================================================================
# 1. Ground-truth components
# ====================================================================
def create_continuum(N, dt):
    """
    Generate a realistic AGN continuum light curve using a
    damped random walk (DRW) process.
    """
    tau_drw = 100.0   # DRW timescale (days)
    sigma_drw = 0.15  # DRW amplitude
    t = np.arange(N) * dt
    x = np.zeros(N)
    for i in range(1, N):
        decay = np.exp(-dt / tau_drw)
        x[i] = decay * x[i-1] + sigma_drw * np.sqrt(1 - decay**2) * np.random.randn()
    # Mean flux ~ 1.0
    continuum = 1.0 + x
    return t, continuum

def create_transfer_function(N, dt, tau_peak, tau_sigma):
    """
    GT transfer function: Gaussian in lag space.
    Ψ(τ) = A * exp(-(τ - τ_peak)^2 / (2σ^2))
    """
    tau = np.arange(N) * dt
    psi = np.exp(-0.5 * ((tau - tau_peak) / tau_sigma)**2)
    psi /= psi.sum() * dt  # normalise to unit integral
    return tau, psi

# ====================================================================
# 2. Forward model
# ====================================================================
def forward_model(continuum, psi, dt, snr):
    """
    L(t) = ∫ Ψ(τ) × C(t-τ) dτ + noise
    Implemented as discrete convolution.
    """
    line_clean = np.convolve(continuum, psi * dt, mode='full')[:len(continuum)]
    noise_std = line_clean.std() / snr
    noise = np.random.normal(0, noise_std, len(continuum))
    line_obs = line_clean + noise
    return line_clean, line_obs, noise_std

# ====================================================================
# 3. Inverse: Tikhonov deconvolution
# ====================================================================
def tikhonov_deconvolve(continuum, line_obs, dt, lam=None):
    """
    Recover Ψ(τ) from:
        L = C * Ψ + noise
    using Non-Negative Least Squares (NNLS) with an explicit
    convolution matrix.  This enforces non-negativity of Ψ(τ)
    and avoids the over-smoothing of frequency-domain Tikhonov.

    We only recover the first M=100 lag values (0–99 days) since
    the GT transfer function is negligible beyond ~50 days.
    """
    N = len(continuum)
    M = min(100, N)  # number of lag bins to recover

    # Build convolution matrix  A[i, j] = C(t_i - τ_j) * dt
    A = np.zeros((N, M))
    for j in range(M):
        for i in range(j, N):
            A[i, j] = continuum[i - j] * dt

    # NNLS solve:  min ||A @ psi - line_obs||^2  s.t. psi >= 0
    psi_nnls, _ = nnls(A, line_obs)

    # Embed into full-length array for downstream compatibility
    psi_rec = np.zeros(N)
    psi_rec[:M] = psi_nnls

    return psi_rec

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
# ====================================================================
# 4. Metrics
# ====================================================================
def compute_metrics(psi_gt, psi_rec, tau, dt):
    """PSNR and CC of recovered transfer function."""
    # Trim to relevant lag range
    max_lag = 80.0
    mask = tau <= max_lag
    gt = psi_gt[mask]
    rec = psi_rec[mask]

    # Normalise both to peak = 1 for a fair, scale-invariant comparison
    gt_peak = gt.max()
    rec_peak = rec.max()
    gt_norm  = gt / gt_peak if gt_peak > 0 else gt
    rec_norm = rec / rec_peak if rec_peak > 0 else rec

    mse = np.mean((gt_norm - rec_norm)**2)
    # data_range = 1.0 after peak-normalisation
    psnr = 10.0 * np.log10(1.0 / mse) if mse > 0 else 100.0
    cc = float(np.corrcoef(gt_norm, rec_norm)[0, 1])
    rmse = float(np.sqrt(mse))

    # Peak lag recovery
    peak_gt = tau[mask][np.argmax(gt)]
    peak_rec = tau[mask][np.argmax(rec)]
    peak_error = abs(peak_rec - peak_gt)

    return psnr, cc, rmse, peak_error, rec_norm, gt_norm

# ====================================================================
# 5. Visualization
# ====================================================================
def plot_results(t, continuum, line_clean, line_obs, tau, psi_gt,
                 psi_rec_norm, ccf_lags, ccf, metrics_dict):
    """4-panel figure."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    max_lag = 80.0
    mask = tau <= max_lag
    ccf_mask = ccf_lags <= max_lag
    
    # Panel 1: Light curves
    ax = axes[0, 0]
    ax.plot(t, continuum, 'b-', lw=0.8, label='Continuum C(t)')
    ax.plot(t, line_obs, 'r-', lw=0.8, alpha=0.6, label='Line L(t) [observed]')
    ax.plot(t, line_clean, 'g--', lw=0.8, label='Line L(t) [clean]')
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Flux")
    ax.set_title("AGN Light Curves")
    ax.legend(fontsize=8)
    
    # Panel 2: Transfer function
    ax = axes[0, 1]
    ax.plot(tau[mask], psi_gt[mask], 'b-', lw=2, label='GT  Ψ(τ)')
    ax.plot(tau[mask], psi_rec_norm, 'r--', lw=2, label='Recovered Ψ(τ)')
    ax.axvline(TAU_PEAK, color='gray', ls=':', lw=1, label=f'True peak = {TAU_PEAK} d')
    ax.set_xlabel("Lag τ (days)")
    ax.set_ylabel("Ψ(τ)")
    ax.set_title(f"Transfer Function | PSNR={metrics_dict['PSNR']:.1f} dB, CC={metrics_dict['CC']:.4f}")
    ax.legend(fontsize=8)
    
    # Panel 3: CCF
    ax = axes[1, 0]
    ax.plot(ccf_lags[ccf_mask], ccf[ccf_mask], 'k-', lw=1.5)
    ax.axvline(TAU_PEAK, color='r', ls='--', lw=1, label=f'True lag = {TAU_PEAK} d')
    peak_ccf_lag = ccf_lags[ccf_mask][np.argmax(ccf[ccf_mask])]
    ax.axvline(peak_ccf_lag, color='b', ls=':', lw=1, label=f'CCF peak = {peak_ccf_lag:.1f} d')
    ax.set_xlabel("Lag τ (days)")
    ax.set_ylabel("CCF")
    ax.set_title("Cross-Correlation Function")
    ax.legend(fontsize=8)
    
    # Panel 4: Residual of transfer function
    ax = axes[1, 1]
    residual = psi_gt[mask] - psi_rec_norm
    ax.plot(tau[mask], residual, 'k-', lw=1)
    ax.axhline(0, color='r', ls='--', lw=0.5)
    ax.set_xlabel("Lag τ (days)")
    ax.set_ylabel("Residual")
    ax.set_title(f"Ψ Residual | Peak error = {metrics_dict['peak_lag_error']:.1f} days")
    ax.fill_between(tau[mask], residual, alpha=0.3, color='gray')
    
    plt.tight_layout()
    for path in [os.path.join(RESULTS_DIR, "vis_result.png"),
                 os.path.join(ASSETS_DIR, "vis_result.png")]:
        fig.savefig(path, dpi=150)
    plt.close(fig)

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **pypetal_reverb**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=25.36 dB ← 🔧 修复前: 2.61 dB, SSIM=N/A (1D))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: pyPETaL: A Pipeline for Estimating AGN Time Lags
- Repository: https://github.com/Zstone19/pypetal
